In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns

Step 1: Load the data

In [5]:
# loading data
test = pd.read_csv("test.csv")
train = pd.read_csv("train.csv")

Step 2: Drop irrelevant coloumns
- PassengerId and Name do not carry an predictive info -> DROP
- Cabin encodes deck, number and side; split later
- keep the other coloumns  

In [6]:
# before dropping Cabin, split it first!
train[['CabinDeck', 'CabinNum', 'Side']] = train['Cabin'].str.split('/', expand=True)
test[['CabinDeck', 'CabinNum', 'Side']] = test['Cabin'].str.split('/', expand=True)

# convert CabinNum to number
train['CabinNum'] = pd.to_numeric(train['CabinNum'], errors='coerce')
test['CabinNum'] = pd.to_numeric(test['CabinNum'], errors='coerce')

# dropping 
train = train.drop(['PassengerId', 'Name', 'Cabin'], axis=1)
test = test.drop(['PassengerId', 'Name', 'Cabin'], axis=1)

Step 3: Handling missing values

In [7]:
# numerical col
num_cols = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck','CabinNum']
for col in num_cols: 
    median_val = train[col].median()
    train[col].fillna(median_val,inplace=True)
    test[col].fillna(median_val,inplace=True)

# categ/bool col
categ_col = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
for col in categ_col: 
    mode_value = train[col].mode()[0]
    train[col].fillna(mode_value,inplace=True)
    test[col].fillna(mode_value,inplace=True)
    

Step 4: Encode categorical variables
- converting to numerical 

In [ ]:
from sklearn.preprocessing import LabelEncoder

le_home = LabelEncoder()
train['HomePlanet'] = le_home.fit_transform(train['HomePlanet'])
test['HomePlanet'] = le_home.transform(test['HomePlanet'])

le_dest = LabelEncoder()
train['Destination'] = le_dest.fit_transform(train['Destination'])
test['Destination'] = le_dest.transform(test['Destination'])

le_cryosleep = LabelEncoder()
train['CryoSleep'] = le_cryosleep.fit_transform(train['CryoSleep'])
test['CryoSleep'] = le_cryosleep.transform(test['CryoSleep'])

le_vip = LabelEncoder()
train['VIP'] = le_vip.fit_transform(train['VIP'])
test['VIP'] = le_vip.transform(test['VIP'])

le_deck = LabelEncoder()
train['CabinDeck'] = le_deck.fit_transform(train['CabinDeck'].astype(str))
test['CabinDeck'] = le_deck.transform(test['CabinDeck'].astype(str))

le_side = LabelEncoder()
train['Side'] = le_side.fit_transform(train['Side'].astype(str))
test['Side'] = le_side.transform(test['Side'].astype(str))

Step 5 & 6: Split data into features and Build the NN model

MLP Classifier 

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

features = ['HomePlanet', 'CryoSleep', 'CabinDeck', 'CabinNum', 'Side', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

result_dict = []

for feature in features:
   
    X = train[[feature]]
    y = train['Transported']

    
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # initialize and train neural network
    nn = MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=42)
    nn.fit(X_train_scaled, y_train)

    # predict on validation set
    y_pred = nn.predict(X_val_scaled)

    # Evaluate
    result_dict.append({'Feature': feature,
                        'Accuracy': accuracy_score(y_val, y_pred),
                        'F1 Score': (f1_score(y_val, y_pred)),
                        'Confusion Matrix': confusion_matrix(y_val, y_pred)
                    })
    

In [46]:
sorted_list = sorted(result_dict, key=lambda x: x['Accuracy'], reverse=True)

print(sorted_list)

[{'Feature': 'CryoSleep', 'Accuracy': 0.7222541690626797, 'F1 Score': 0.6811881188118811, 'Confusion Matrix': array([[740, 121],
       [362, 516]], dtype=int64)}, {'Feature': 'RoomService', 'Accuracy': 0.6440483036227717, 'F1 Score': 0.7220475976650202, 'Confusion Matrix': array([[316, 545],
       [ 74, 804]], dtype=int64)}, {'Feature': 'Spa', 'Accuracy': 0.6319723979298447, 'F1 Score': 0.711971197119712, 'Confusion Matrix': array([[308, 553],
       [ 87, 791]], dtype=int64)}, {'Feature': 'VRDeck', 'Accuracy': 0.6308223116733755, 'F1 Score': 0.7118491921005385, 'Confusion Matrix': array([[304, 557],
       [ 85, 793]], dtype=int64)}, {'Feature': 'ShoppingMall', 'Accuracy': 0.6175963197239793, 'F1 Score': 0.7016599371915657, 'Confusion Matrix': array([[292, 569],
       [ 96, 782]], dtype=int64)}, {'Feature': 'FoodCourt', 'Accuracy': 0.5934445083381253, 'F1 Score': 0.6847971466785555, 'Confusion Matrix': array([[264, 597],
       [110, 768]], dtype=int64)}, {'Feature': 'HomePlanet', 

In [57]:

best_n = len(sorted_list)  # change this based on your results
best_features = [x['Feature'] for x in sorted_list[:best_n]]
print("Best features for MLP Classifier:", best_features)

X = train[best_features]
y = train['Transported']

    
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# initialize and train neural network
nn = MLPClassifier(hidden_layer_sizes=(50,), max_iter=500, random_state=42)
nn.fit(X_train_scaled, y_train)

# predict on validation set
y_pred = nn.predict(X_val_scaled)

# Evaluate
print(f'Accuracy: {accuracy_score(y_val, y_pred)}')
print(f'F1 Score: {(f1_score(y_val, y_pred))}')
print(f'Confusion Matrix: {confusion_matrix(y_val, y_pred)}')
            

Best features for MLP Classifier: ['CryoSleep', 'RoomService', 'Spa', 'VRDeck', 'ShoppingMall', 'FoodCourt', 'HomePlanet', 'CabinDeck', 'CabinNum', 'Age', 'Destination', 'Side', 'VIP']
Accuracy: 0.7924094307073031
F1 Score: 0.79639029892837
Confusion Matrix: [[672 189]
 [172 706]]


Random Forest implementation 

Feature one by one 

In [49]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

features = ['HomePlanet', 'CryoSleep', 'CabinDeck', 'CabinNum', 'Side', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

result_dict2 = []

for feature in features:
   
    X = train[[feature]]
    y = train['Transported']

    
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # initialize and train neural network
    rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
    rf.fit(X_train, y_train)

    # predict on validation set
    y_pred = rf.predict(X_val)

    # Evaluate
    result_dict2.append({'Feature': feature,
                        'Accuracy': accuracy_score(y_val, y_pred),
                        'F1 Score': (f1_score(y_val, y_pred)),
                        'Confusion Matrix': confusion_matrix(y_val, y_pred),
                        'Importance': rf.feature_importances_[0] 
                    })

print(f"{feature} importance: {rf.feature_importances_[0]:.4f}")

VRDeck importance: 1.0000


In [50]:
sorted_list2 = sorted(result_dict2, key=lambda x: x['Accuracy'], reverse=True)

print(sorted_list2)

[{'Feature': 'CryoSleep', 'Accuracy': 0.7222541690626797, 'F1 Score': 0.6811881188118811, 'Confusion Matrix': array([[740, 121],
       [362, 516]], dtype=int64), 'Importance': 1.0}, {'Feature': 'RoomService', 'Accuracy': 0.6716503737780334, 'F1 Score': 0.7274463007159905, 'Confusion Matrix': array([[406, 455],
       [116, 762]], dtype=int64), 'Importance': 1.0}, {'Feature': 'Spa', 'Accuracy': 0.6624496837262794, 'F1 Score': 0.7060590886329494, 'Confusion Matrix': array([[447, 414],
       [173, 705]], dtype=int64), 'Importance': 1.0}, {'Feature': 'VRDeck', 'Accuracy': 0.6584243818286372, 'F1 Score': 0.7076771653543308, 'Confusion Matrix': array([[426, 435],
       [159, 719]], dtype=int64), 'Importance': 1.0}, {'Feature': 'ShoppingMall', 'Accuracy': 0.648073605520414, 'F1 Score': 0.7126760563380281, 'Confusion Matrix': array([[368, 493],
       [119, 759]], dtype=int64), 'Importance': 1.0}, {'Feature': 'FoodCourt', 'Accuracy': 0.6187464059804485, 'F1 Score': 0.6931975937066173, 'Conf

In [55]:
best_n = len(sorted_list2)  # change this based on your results
best_features = [x['Feature'] for x in sorted_list2[:best_n]]
print("Best features for RandomForest Classifier :", best_features)

X = train[best_features]
y = train['Transported']

    
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# initialize and train random forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

# predict on validation set
y_pred = rf.predict(X_val)

# Evaluate
print(f'Accuracy: {accuracy_score(y_val, y_pred)}')
print(f'F1 Score: {(f1_score(y_val, y_pred))}')
print(f'Confusion Matrix: {confusion_matrix(y_val, y_pred)}')
            

Best features for RandomForest Classifier : ['CryoSleep', 'RoomService', 'Spa', 'VRDeck', 'ShoppingMall', 'FoodCourt', 'HomePlanet', 'CabinDeck', 'CabinNum', 'Destination', 'Side', 'Age', 'VIP']
Accuracy: 0.7952846463484762
F1 Score: 0.8026607538802661
Confusion Matrix: [[659 202]
 [154 724]]


Training all features at once in RandomForestClassifier 

In [58]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

features = ['HomePlanet', 'CryoSleep', 'CabinDeck', 'CabinNum', 'Side', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

X = train[best_features]
y = train['Transported']
   
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# training ALL features to get the real importance
rf_all = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
rf_all.fit(X_train, y_train)

feature_importances = dict(zip(features,rf_all.feature_importances_))

result_dict3 = []

for feature in features: 
   
    X = train[[feature]]
    y = train['Transported']

    
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42)
    rf.fit(X_train,y_train)

    # predict on validation set
    y_pred = rf.predict(X_val)

    # Evaluate
    result_dict3.append({'Feature': feature,
                        'Accuracy': accuracy_score(y_val, y_pred),
                        'F1 Score': (f1_score(y_val, y_pred)),
                        'Confusion Matrix': confusion_matrix(y_val, y_pred),
                        'Importance': feature_importances[feature]
                    })

print(f"{feature} importance: {rf.feature_importances_[0]:.4f}")

VRDeck importance: 1.0000


In [59]:
sorted_list3 = sorted(result_dict3, key=lambda x: x['Accuracy'], reverse=True)

print(sorted_list3)

[{'Feature': 'CryoSleep', 'Accuracy': 0.7222541690626797, 'F1 Score': 0.6811881188118811, 'Confusion Matrix': array([[740, 121],
       [362, 516]], dtype=int64), 'Importance': 0.11458729067254818}, {'Feature': 'RoomService', 'Accuracy': 0.6716503737780334, 'F1 Score': 0.7274463007159905, 'Confusion Matrix': array([[406, 455],
       [116, 762]], dtype=int64), 'Importance': 0.14090242458029276}, {'Feature': 'Spa', 'Accuracy': 0.6624496837262794, 'F1 Score': 0.7060590886329494, 'Confusion Matrix': array([[447, 414],
       [173, 705]], dtype=int64), 'Importance': 0.02365078142152882}, {'Feature': 'VRDeck', 'Accuracy': 0.6584243818286372, 'F1 Score': 0.7076771653543308, 'Confusion Matrix': array([[426, 435],
       [159, 719]], dtype=int64), 'Importance': 0.0022942747417887435}, {'Feature': 'ShoppingMall', 'Accuracy': 0.648073605520414, 'F1 Score': 0.7126760563380281, 'Confusion Matrix': array([[368, 493],
       [119, 759]], dtype=int64), 'Importance': 0.021100337663495405}, {'Feature':

In [64]:
best_n = len(sorted_list3) # change this based on your results
best_features = [x['Feature'] for x in sorted_list3[:best_n]]
print("Best features for RandomForest Classifier :", best_features)

X = train[best_features]
y = train['Transported']

    
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# initialize and train random forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

# predict on validation set
y_pred = rf.predict(X_val)

# Evaluate
print(f'Accuracy: {accuracy_score(y_val, y_pred)}')
print(f'F1 Score: {(f1_score(y_val, y_pred))}')
print(f'Confusion Matrix: {confusion_matrix(y_val, y_pred)}')
            

Best features for RandomForest Classifier : ['CryoSleep', 'RoomService', 'Spa', 'VRDeck', 'ShoppingMall', 'FoodCourt', 'HomePlanet', 'CabinDeck', 'CabinNum', 'Destination', 'Side', 'Age', 'VIP']
Accuracy: 0.7952846463484762
F1 Score: 0.8026607538802661
Confusion Matrix: [[659 202]
 [154 724]]


Conclusion for random forest classifier: Since accuracy is the same, it shows that the features are independently useful and do not rely on each other